# 05 — Ajuste de hiperparámetros y segunda campaña

Tareas **4.5** (ajuste y experimentación) y **4.6** (segunda campaña con la
configuración optimizada) del WBS.

**De dónde salen los experimentos.** No es un barrido a ciegas: la primera
campaña ya acotó el espacio, y cada configuración cuesta horas de GPU sobre
un presupuesto fijo de Colab.

| Lo que mostró la campaña 1 | Lo que implica |
|---|---|
| Sobreajusta desde la época 14: la pérdida de entrenamiento cayó 42 % y la validación no se movió | El margen está en **regularización**, no en capacidad ni en más épocas |
| Picos de validación en las épocas 2, 4 y 10 con el LR cerca del máximo | Probar **calentamiento** del learning rate |
| Stockfish d12 vs d20 difieren RMSE 0,057 contra los 0,261 del modelo | El ruido de etiqueta explica <5 % del error: **hay margen real** |

**Referencias a batir:** piso de material 0,3973 · campaña 1 en validación **0,2584**.


## 1. Entorno


In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean los 17
# tests de integracion del pipeline, que son la evidencia del requerimiento 3.2.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
import numpy as np
import torch
from chessdl.colab import TRAINING, describe_runtime

runtime = describe_runtime(phase=TRAINING)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
for aviso in runtime.warnings():
    print("AVISO:", aviso)

## 2. Tests


In [ ]:
!{sys.executable} -m pytest -q

## 3. Dataset, partición, caché y pisos

Idéntico a la notebook 04. Si el caché ya está en disco no se reconstruye.


In [ ]:
from chessdl.config import load_config
from chessdl import hf
from chessdl.data import schema
from chessdl.training.split import describe_split, leaked_games, split_masks
from chessdl.training.cache import build_cache, cache_path_for, load_cache
from chessdl.training.baselines import material_baseline, mean_baseline

cfg = load_config()
token = hf.get_token()
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")
tabla = schema.read_dataset(schema.shard_paths(directorio))

fens     = tabla["fen"].to_pylist()
game_ids = tabla["game_id"].to_pylist()
targets  = np.asarray(tabla["value_stm"], dtype=np.float32)

split_cfg = cfg.training.split_config()
masks = split_masks(game_ids, split_cfg)
assert not leaked_games(masks, game_ids)

ruta_cache = cache_path_for(cfg.training.cache_dir)
if not ruta_cache.exists():
    build_cache(fens, ruta_cache, progress=True)
cache = load_cache(ruta_cache, expected_rows=len(fens))

idx_train = np.flatnonzero(masks['train'])
idx_val   = np.flatnonzero(masks['val'])
idx_test  = np.flatnonzero(masks['test'])

rng = np.random.default_rng(0)
sub = np.sort(rng.choice(idx_train, size=min(200_000, len(idx_train)), replace=False))
media = mean_baseline(targets[idx_train], targets[idx_val])
material, _ = material_baseline(np.asarray(cache[sub]), targets[sub],
                                np.asarray(cache[idx_val]), targets[idx_val])
pisos = {'media': media.rmse, 'material': material.rmse}
print(media); print(material)

## 4. El barrido (tarea 4.5)

Cinco brazos. `base` es el control —la misma configuración de la campaña 1
pero con el coseno sobre el presupuesto corto— y no es una corrida
desperdiciada: anelar a cero en 15 épocas es una trayectoria distinta y mejor
terminada que frenar una de 30, y sin ella los otros cuatro no tienen contra
qué compararse al mismo presupuesto.

**Costo: ~4,7 h de GPU** (5 × 15 épocas × ~227 s). Si querés recortar, sacá
un brazo de la tupla o bajá `EPOCAS_BARRIDO`.

> **Es reanudable.** Cada brazo tiene su propio nombre de corrida en el Hub.
> Si Colab corta a mitad del barrido, se vuelve a correr esta celda: los
> brazos terminados se detectan y se saltean, y el que quedó a medias sigue
> desde su última época.


In [ ]:
from chessdl.training.experiments import DEFAULT_SWEEP, run_sweep

for e in DEFAULT_SWEEP:
    print(f'{e.name:<10}{e.rationale}')

In [ ]:
from chessdl.models.resnet import ResNetConfig

EPOCAS_BARRIDO = 15
REFERENCIA = 0.2584   # mejor validacion de la campana 1

arquitectura = ResNetConfig(channels=128, blocks=8)
repo_modelos = f'{cfg.output.hf_namespace}/{cfg.training.hf_models_repo}'

barrido = run_sweep(
    DEFAULT_SWEEP, cache, targets, idx_train, idx_val,
    base_model=arquitectura,
    repo_id=repo_modelos,
    local_dir='/content/ceia-chess/checkpoints',
    token=token,
    push_to_hub=cfg.training.push_to_hub and token is not None,
    epochs=EPOCAS_BARRIDO,
    prefix='sweep',
    batch_size=cfg.training.batch_size,
    learning_rate=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
    loss_name=cfg.training.loss,
    seed=cfg.training.split_seed,
    baselines=pisos,
    reference=REFERENCIA,
)

## 5. Resultados del barrido


In [ ]:
print(barrido.table())

In [ ]:
import matplotlib.pyplot as plt
from chessdl import viz

viz.apply_style()
fig, ax = plt.subplots()
for i, (nombre, run) in enumerate(barrido.runs.items()):
    ep = [e.epoch for e in run.history.epochs]
    ax.plot(ep, [e.val_rmse for e in run.history.epochs],
            marker='o', markersize=3, color=viz.SERIES[i % len(viz.SERIES)], label=nombre)
ax.axhline(REFERENCIA, color=viz.INK_SECONDARY, linestyle='--', linewidth=1.5)
ax.text(1, REFERENCIA, '  campana 1', color=viz.INK_SECONDARY, fontsize=8, va='bottom')
ax.legend(loc='upper right', ncol=2)
viz.label_axes(ax, 'Los cinco brazos del barrido', 'epoca', 'RMSE sobre validacion',
               note='Al mismo presupuesto de epocas, para que la comparacion sea entre configuraciones')
fig.tight_layout()

In [ ]:
# El sobreajuste, que es lo que el barrido intenta atacar.
fig, ax = plt.subplots()
for i, (nombre, run) in enumerate(barrido.runs.items()):
    ep  = [e.epoch for e in run.history.epochs]
    raz = [e.val_rmse ** 2 / e.train_loss for e in run.history.epochs]
    ax.plot(ep, raz, marker='o', markersize=3, color=viz.SERIES[i % len(viz.SERIES)], label=nombre)
ax.axhline(1.0, color=viz.INK_MUTED, linestyle=':', linewidth=1)
ax.legend(loc='upper left', ncol=2)
viz.label_axes(ax, 'Cuanto se separa la validacion del entrenamiento', 'epoca',
               'error de validacion / perdida de entrenamiento',
               note='En la campana 1 esta razon llego a 2,16 en la epoca 30')
fig.tight_layout()

## 6. Segunda campaña (tarea 4.6)

El ganador del barrido, ahora a **18 épocas**.

Por qué se confirma en vez de adoptarse directo: **cribar regularización con
presupuesto corto no es neutral.** Menos entrenamiento favorece a menos
regularización, así que un `weight_decay` o un dropout que gane a 15 épocas
puede quedarse corto a 18. La tabla del barrido ordena candidatos; el número
que va a la memoria sale de esta campaña.


In [ ]:
from chessdl.training.loop import seed_everything
from chessdl.models.resnet import ChessResNet
from chessdl.training.checkpoint import HubCheckpoints
from chessdl.training.loop import train

EPOCAS_FINAL = 18

ganador = barrido.experiments[barrido.best_name()]
print(f'Ganador del barrido: {ganador.name} -- {ganador.rationale}')
print(f'Cambia: {ganador.overrides()}' + (f" + dropout {ganador.dropout}" if ganador.dropout else ''))

config_final = ganador.model_config(arquitectura)
# La siembra va ANTES de construir el modelo: train() tambien siembra,
# pero para entonces los pesos iniciales ya salieron del estado en que
# estuviera el interprete, y no serian reproducibles entre sesiones.
seed_everything(cfg.training.split_seed)
modelo = ChessResNet(config_final)
print(modelo.describe())

In [ ]:
checkpoints = HubCheckpoints(
    repo_id=repo_modelos,
    run_name=f'campana2-{ganador.name}',
    local_dir='/content/ceia-chess/checkpoints',
    token=token,
    enabled=cfg.training.push_to_hub and token is not None,
)

ajustes = dict(
    epochs=EPOCAS_FINAL,
    batch_size=cfg.training.batch_size,
    learning_rate=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
    loss_name=cfg.training.loss,
    seed=cfg.training.split_seed,
)
ajustes.update(ganador.overrides())
ajustes['epochs'] = EPOCAS_FINAL   # el presupuesto lo fija la campana, no el brazo

campana2 = train(
    modelo, cache, targets, idx_train, idx_val,
    checkpoints=checkpoints,
    model_config={'channels': config_final.channels, 'blocks': config_final.blocks,
                  'dropout': config_final.dropout},
    baselines=pisos,
    **ajustes,
)
print()
print(campana2.summary())

## 7. Evaluación final sobre test

El split de test se toca **una sola vez**, con el mejor checkpoint por
validación de esta segunda campaña.


In [ ]:
from chessdl.training.checkpoint import BEST_NAME, load_checkpoint
from chessdl.training.loop import evaluate_split

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
mejor = checkpoints.fetch(BEST_NAME)
if mejor is not None:
    load_checkpoint(mejor, modelo, map_location=dispositivo)
modelo = modelo.to(dispositivo)

test2 = evaluate_split(modelo, cache, targets, idx_test, dispositivo)
print('TEST -- campana 2')
print(test2.summary())

## 8. Resumen: qué ganó el ajuste


In [ ]:
TEST_CAMPANA1 = 0.2609   # RMSE de test de la notebook 04

print(f"{'':<34}{'RMSE':>10}{'R2':>10}")
print('-' * 56)
for nombre, valor in [('media constante', pisos['media']),
                      ('material lineal', pisos['material']),
                      ('ResNet campana 1 (test)', TEST_CAMPANA1),
                      ('ResNet campana 2 (test)', test2.rmse)]:
    print(f"{nombre:<34}{valor:>10.4f}{1-(valor/pisos['media'])**2:>10.3f}")
print()
delta = (TEST_CAMPANA1 - test2.rmse) / TEST_CAMPANA1
print(f"El ajuste {'mejoro' if delta > 0 else 'EMPEORO'} el test un {abs(delta):.2%}.")
if abs(delta) < 0.01:
    print('Menos del 1 %: el ajuste de hiperparametros no es donde estaba el margen.')
    print('Eso tambien es un resultado, y apunta a los datos como siguiente palanca:')
    print('se consumio el 47 % del extracto, y el ruido de etiqueta explica <5 % del error.')
print()
print(f'Pesos y metricas: https://huggingface.co/{repo_modelos}')